# Neuronale Netze 12: Ein erstes PyTorch-Netz

## Was machen wir heute?

In Colab 11 hast du Bilder und Klassen von **Fashion-MNIST** untersucht. Jetzt bauen wir ein kleines Netz für dieselben Bilder.

Du wirst sehen, was die beiden Teile einer Python-Klasse tun: **`__init__`** baut das Netz auf und **`forward`** berechnet seine Ausgabe. Für ein Bild liefert das Netz **zehn Zahlen**: eine Zahl pro Klasse. Wir vergleichen die Ausgabe **vor und nach dem Training**.

Du musst den Programmcode **nicht** selbst schreiben. Lies jeweils die kurze Aufgabe, führe die Zelle aus und trage deine Antwort im **Leitprogramm** ein.


## Start: So arbeitest du

1. Führe die Zellen mit **▶** der Reihe nach aus. Warte jeweils, bis die Zelle fertig ist.
2. Beim ersten Start lädt Colab die Bilder herunter. Das kann etwas dauern.
3. **Ändere keinen Code.** Erst in E6 änderst du eine Bildnummer im Eingabefeld.
4. Schreibe die Antworten in das separate Leitprogramm. Bei einem Fehler: Starte mit **Laufzeit → Laufzeit neu starten und alle ausführen** nochmals von oben.

Die Bilder haben **28 × 28 Pixel**. Das Netz nutzt für das Training nur einen Teil der Bilder, damit der Versuch rasch läuft. Wir verwenden die **CPU**; eine GPU ist nicht nötig.


In [ ]:
#@title Start – Daten und Werkzeuge bereitstellen { display-mode: "form" }
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import FashionMNIST
from torchvision.transforms import ToTensor
import matplotlib.pyplot as plt

torch.manual_seed(7)

class_names = [
    "T-Shirt/Oberteil", "Hose", "Pullover", "Kleid", "Mantel",
    "Sandale", "Hemd (Shirt)", "Turnschuh", "Tasche", "Stiefelette",
]

train_dataset = FashionMNIST(
    root="/content/fashion_mnist_data", train=True, download=True,
    transform=ToTensor(),
)
test_dataset = FashionMNIST(
    root="/content/fashion_mnist_data", train=False, download=True,
    transform=ToTensor(),
)

# Diese kleinen Ausschnitte halten die Wartezeit kurz.
train_subset = Subset(train_dataset, range(8000))
test_subset = Subset(test_dataset, range(500))
train_loader = DataLoader(
    train_subset, batch_size=128, shuffle=True,
    generator=torch.Generator().manual_seed(7), num_workers=0,
)
test_loader = DataLoader(test_subset, batch_size=128, shuffle=False, num_workers=0)


def count_correct(current_model, data_loader):
    current_model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in data_loader:
            predictions = current_model(images).argmax(dim=1)
            correct += int((predictions == labels).sum().item())
    return correct


def show_prediction(current_model, image_index):
    if not 0 <= image_index < len(train_dataset):
        print("Bitte eine Bildnummer von 0 bis 59'999 eingeben.")
        return
    image, true_id = train_dataset[image_index]
    current_model.eval()
    with torch.no_grad():
        scores = current_model(image.unsqueeze(0))[0]
    predicted_id = int(scores.argmax().item())

    print(f"Trainingsbild {image_index}")
    print(f"Wahres Label: {true_id} – {class_names[true_id]}")
    print(f"Zehn Rohwerte (einer pro Klasse): {len(scores)}")
    for class_id, score in enumerate(scores):
        print(f"  {class_id}: {class_names[class_id]:<18} {score.item():+7.2f}")
    print(f"Grösster Wert bei ID {predicted_id} → Vorhersage: {class_names[predicted_id]}")
    if predicted_id == true_id:
        print("Vorhersage und wahres Label sind gleich.")
    else:
        print("Vorhersage und wahres Label sind verschieden.")

    plt.figure(figsize=(3.2, 3.2))
    plt.imshow(image.squeeze(0), cmap="gray", vmin=0, vmax=1)
    plt.title(f"Trainingsbild {image_index}")
    plt.axis("off")
    plt.show()
    return scores.detach().clone(), predicted_id, true_id


print("\nDie Daten sind bereit. Gerät: CPU.")


## Experiment E1: Welche Bilder nutzen wir?

Führe die nächste Zelle aus. Die ersten zwei Zahlen zeigen nochmals den ganzen Datensatz. Für **E1 im Leitprogramm** brauchst du nur die kleine Auswahl: **Wie viele Trainingsbilder und Testbilder verwenden wir hier?** Training ist zum Lernen. Mit anderen Bildern im Testteil prüfen wir das Netz. Schreibe auch den Zweck des Testteils auf.

Ein kleiner Teil reicht für unseren ersten Versuch. Die Bildnummern **42** und **43** liegen im verwendeten Trainingsteil.


In [ ]:
#@title E1 – Bildmengen anzeigen { display-mode: "form" }
print(f"Alle Trainingsbilder: {len(train_dataset):,}".replace(",", "'"))
print(f"Alle Testbilder: {len(test_dataset):,}".replace(",", "'"))
print(f"Für dieses Training: {len(train_subset):,}".replace(",", "'"))
print(f"Für unsere kleine Testzählung: {len(test_subset):,}".replace(",", "'"))


## Experiment E2: Die Klasse für unser Netz

Lies die kurze Klasse unten. Führe die Zelle dann **einmal** aus. Du musst nichts verändern.

- **`__init__`** läuft, wenn `FashionNet()` ein Netz erstellt. Hier stehen die Bausteine des Netzes.
- **`forward`** beschreibt den Weg eines Bildes durch diese Bausteine. Er wird benutzt, wenn wir später `model(images)` aufrufen.
- **`self.output = nn.Linear(64, 10)`** erstellt den letzten Baustein. Die **10** steht für unsere zehn Klassen.

Suche im Code die Zeile mit **`28 * 28`** und die Zeile, die das Ergebnis zurückgibt. Trage deine Beobachtungen bei **E2** ein.

**Achtung:** Wenn du diese Zelle später nochmals ausführst, entsteht wieder ein neues, untrainiertes Netz. Starte dann E3, E4 und E5 nochmals in dieser Reihenfolge.


In [ ]:
# E2 – Netzklasse lesen und starten (bitte nichts ändern)
class FashionNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.hidden = nn.Linear(28 * 28, 64)
        self.relu = nn.ReLU()
        self.output = nn.Linear(64, 10)

    def forward(self, images):
        pixels = self.flatten(images)
        middle = self.relu(self.hidden(pixels))
        scores = self.output(middle)
        return scores


model = FashionNet()
print(model)

## Experiment E3: Zehn Ausgaben vor dem Training

Führe die nächste Zelle aus. Sie zeigt **Trainingsbild 42**, das du aus Colab 11 kennst. Das Netz hat **noch nicht gelernt**.

Neben jeder Klassen-ID steht eine Zahl. Diese Zahlen heissen hier **Rohwerte**. Sie sind **keine Prozentzahlen**. Das Netz nimmt die Klasse mit dem **grössten Rohwert** als Vorhersage. Ein Rohwert darf auch negativ sein.

Suche den grössten Wert. Vergleiche **Vorhersage** und **wahres Label**. Trage deine Beobachtungen bei **E3** ein. Es ist normal, wenn ein untrainiertes Netz falsch liegt.


In [ ]:
#@title E3 – Bild 42 vor dem Training untersuchen { display-mode: "form" }
before_scores, before_predicted_id, true_id = show_prediction(model, 42)


## Experiment E4: Das Netz kurz trainieren

Führe die nächste Zelle **einmal** aus und warte, bis **Runde 2** erscheint. Das Training braucht etwas Zeit.

Beim Lernen vergleicht PyTorch die Ausgaben mit den richtigen Labels und verändert Zahlen im Netz. **Verlust** zeigt, wie schlecht die Ausgaben zur Lösung passen. Ein **kleinerer** mittlerer Verlust ist hier ein gutes Zeichen. Verlust bedeutet **nicht** «Anzahl falscher Bilder».

Lies die Werte für Runde 1 und Runde 2 ab. Trage sie bei **E4** ein. Die genauen Dezimalzahlen können etwas abweichen.


In [ ]:
#@title E4 – Zwei Runden trainieren { display-mode: "form" }
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(2):
    model.train()
    loss_sum = 0.0
    for images, labels in train_loader:
        scores = model(images)
        loss = loss_fn(scores, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * len(images)

    mean_loss = loss_sum / len(train_subset)
    print(f"Runde {epoch + 1}: mittlerer Verlust = {mean_loss:.3f}")


## Experiment E5: Dasselbe Bild nach dem Training

Führe die nächste Zelle aus. Sie zeigt **nochmals Trainingsbild 42** und die zehn neuen Rohwerte. Vergleiche die Vorhersage mit E3. Hat sich etwas geändert? Ist die Vorhersage jetzt gleich wie das wahre Label?

Die erste Zeile zählt zusätzlich richtige Vorhersagen für **500 andere Bilder** aus dem Testteil. Lies ab, wie viele davon nach dem Training richtig erkannt wurden. Ein einzelnes Bild kann auch nach dem Training falsch bleiben.

Trage die Antworten bei **E5** ein.


In [ ]:
#@title E5 – Bild 42 nach dem Training untersuchen { display-mode: "form" }
after_correct = count_correct(model, test_loader)
print(f"Nach dem Training: {after_correct} von 500 Testbildern richtig.")
print()
after_scores, after_predicted_id, true_id = show_prediction(model, 42)
print(f"Vorhersage für Bild 42 geändert: {'Ja' if after_predicted_id != before_predicted_id else 'Nein'}")


## Experiment E6: Ein zweites Bild prüfen

Unten steht zuerst **42**. Führe die Zelle aus. Ändere dann im Eingabefeld **nur die Bildnummer von 42 zu 43** und führe **dieselbe Zelle** nochmals aus.

Bild 42 und Bild 43 haben unterschiedliche wahre Labels. Lies für Bild 43 das wahre Label und die Vorhersage ab. Sind beide **gleich** oder **verschieden**? Trage die Antworten bei **E6** ein.

**Wichtig:** Nur das Eingabefeld ändern. Den Programmcode nicht verändern.


In [ ]:
#@title E6 – Bildnummer 42 und 43 vergleichen { display-mode: "form" }
image_index = 42 #@param {type:"integer"}
show_prediction(model, image_index)


## Schluss

Beantworte die **Schlussfrage** im Leitprogramm: Warum beweisen ein oder zwei richtige Vorhersagen noch nicht, dass das Netz insgesamt gut ist? Denke an die Zählung mit den 500 Testbildern.

Eine Vorhersage ist **nicht immer richtig**. Wir können sie mit dem wahren Label vergleichen. Deine Werte dürfen leicht von den Werten anderer Lernender abweichen.

Fertig? Speichere **das ausgefüllte Leitprogramm** und gib es wie besprochen ab. Das Colab musst du nicht abgeben.


Copyright 2026 [Markus Ineichen](mailto:markus.ineichen1@sluz.ch)

Code licence: [MIT License](https://mit-license.org/)

Text license: [Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International](https://creativecommons.org/licenses/by-nc-sa/4.0/)